# Applying Regex to Parse Clinical Reports

## Project Overview

This project focuses on using advanced Python Regular Expressions (Regex) to extract multiple structured fields from realistic and messy clinical reports.

The parser extracts patient information, vital signs, laboratory results, diagnoses, and medications using named groups, alternation, lookaheads, lookbehinds, and flexible Regex patterns. The extracted information is then organized into structured records and converted into a Pandas DataFrame.

The project also tests the parser on multiple clinical reports with different formatting styles and performs final validation to ensure the extracted data is complete and accurate.

## Key Concepts

* Named Groups
* Alternation
* Positive Lookahead
* Positive Lookbehind
* Flexible Regex Patterns
* re.search() and re.finditer()
* Structured Data Extraction
* Clinical Text Parsing
* Pandas DataFrame Conversion
* Data Validation


## Step 1  Import Required Libraries

In [2]:
import re
import pandas as pd

print('All libararies are imported')


All libararies are imported


## Step 2 Create a Realistic Clinical Report

Now we’ll create messy but realistic clinical text. The goal is to make our regex handle different separators, units, and formatting rather than relying on perfectly clean text.

In [3]:
clinical_report = """
PATIENT INFORMATION
Patient: John Smith
MRN: PT-20481
Age: 47 years | Sex: Male

VITAL SIGNS
BP 138/88 mmHg
Heart Rate: 82 bpm
Temperature = 98.6 F
Respiratory Rate: 18/min

LABORATORY RESULTS
Fasting Glucose = 126 mg/dL
HbA1c: 6.7 %
Total Cholesterol: 212 mg/dL

CLINICAL ASSESSMENT
Diagnosis: Type 2 Diabetes Mellitus
Status: Stable

MEDICATIONS
Metformin 500 mg twice daily
Atorvastatin 20 mg once daily

Follow-up recommended in 3 months.
"""

print(clinical_report)


PATIENT INFORMATION
Patient: John Smith
MRN: PT-20481
Age: 47 years | Sex: Male

VITAL SIGNS
BP 138/88 mmHg
Heart Rate: 82 bpm
Temperature = 98.6 F
Respiratory Rate: 18/min

LABORATORY RESULTS
Fasting Glucose = 126 mg/dL
HbA1c: 6.7 %
Total Cholesterol: 212 mg/dL

CLINICAL ASSESSMENT
Diagnosis: Type 2 Diabetes Mellitus
Status: Stable

MEDICATIONS
Metformin 500 mg twice daily
Atorvastatin 20 mg once daily

Follow-up recommended in 3 months.



## Step 3 Basic Clinical Field Extraction

In [4]:
patient_pattern = r"Patient:\s*(?P<patient_name>[A-Za-z ]+)"

match = re.search(patient_pattern, clinical_report)

if match:
    print(match.group("patient_name"))

John Smith


## Step 3.1  Extract MRN

Now extract the medical record number using another named groups

In [5]:
mrn_pattern = r"MRN:\s*(?P<mrn>[A-Z]{2}-\d+)"

match = re.search(mrn_pattern, clinical_report)

if match:
    print(match.group("mrn"))

PT-20481


## Step 4 Extract Patient Demographics with Named Groups

In [6]:
patient_pattern = (
    r"Patient:\s*(?P<patient_name>[A-Za-z ]+)\s*"
    r"MRN:\s*(?P<mrn>[A-Z]{2}-\d+)\s*"
    r"Age:\s*(?P<age>\d+)\s*years\s*\|\s*"
    r"Sex:\s*(?P<sex>[A-Za-z]+)"
)

match = re.search(patient_pattern, clinical_report)

if match:
    print(match.groupdict())

{'patient_name': 'John Smith', 'mrn': 'PT-20481', 'age': '47', 'sex': 'Male'}


## Step 5  Extract Vital Signs

Now we'll extract four clinical measurements:

1. Blood pressure = 138/88
2. Heart rate = 82
3. Temperature = 98.6
4. Respiratory rate = 18

This time we'll make the regex more flexible because clinical reports often use different separators such as :, =, or just spaces.

### 5.1 Blood Pressure

In [7]:
bp_pattern = r"BP\s*[:=]?\s*(?P<bp>\d{2,3}/\d{2,3})\s*mmHg"

match = re.search(bp_pattern, clinical_report)

if match:
    print(match.groupdict())

{'bp': '138/88'}


    Pattern breakdown
    1. BP                   Blood pressure label
    2. \s*                  Optional spaces
    3. [:=]?                Optional ":" or "="
    4. \s*                  Optional spaces
    5. (?P<bp>...)          Named group
    6. \d{2,3}/\d{2,3}      Example: 138/88
    7. mmHg                 Blood-pressure unit

### 5.2 Heart Rate

In [8]:
hr_pattern = r"Heart Rate\s*[:=]?\s*(?P<heart_rate>\d{2,3})\s*bpm"

match = re.search(hr_pattern, clinical_report)

if match:
    print(match.groupdict())

{'heart_rate': '82'}


### 5.3 Temperature

Here we'll allow both an integer and decimal temperature

In [9]:
temp_pattern = r"Temperature\s*[:=]?\s*(?P<temperature>\d+(?:\.\d+)?)\s*F"

match = re.search(temp_pattern, clinical_report)

if match:
    print(match.groupdict())

{'temperature': '98.6'}


### 5.4 Respiratory Rate

In [10]:
rr_pattern = r"Respiratory Rate\s*[:=]?\s*(?P<respiratory_rate>\d{1,2})\s*/min"

match = re.search(rr_pattern, clinical_report)

if match:
    print(match.groupdict())

{'respiratory_rate': '18'}


### 5.5 blood pressure

In [11]:
bp_pattern = r"BP\s*[:=]?\s*(?P<bp>\d{2,3}/\d{2,3})\s*mmHg"

match = re.search(bp_pattern, clinical_report)

if match:
    print(match.groupdict())
else:
    print("Blood pressure not found")

{'bp': '138/88'}


## Step 6  Extract Laboratory Results Using Alternation

Now we'll use alternation (|) so one regex can recognize multiple laboratory tests.

In [12]:
lab_pattern = (
    r"(?P<test>Fasting Glucose|HbA1c|Total Cholesterol)"
    r"\s*[:=]\s*"
    r"(?P<value>\d+(?:\.\d+)?)"
    r"\s*(?P<unit>mg/dL|%)"
)

matches = re.finditer(lab_pattern, clinical_report)

for match in matches:
    print(match.groupdict())

{'test': 'Fasting Glucose', 'value': '126', 'unit': 'mg/dL'}
{'test': 'HbA1c', 'value': '6.7', 'unit': '%'}
{'test': 'Total Cholesterol', 'value': '212', 'unit': 'mg/dL'}


## Step 7  Lookahead for Context-Aware Extraction

Now we introduce one of the advanced regex concepts from the task: positive lookahead.

A lookahead checks what comes after the current position without including that text in the match.

For example, we can find a number only when it is followed by mg/dL.

In [13]:
glucose_pattern = r"(?P<glucose>\d+(?:\.\d+)?)(?=\s*mg/dL)"

match = re.search(glucose_pattern, clinical_report)

if match:
    print(match.groupdict())

{'glucose': '126'}


### Step 8  Positive Lookbehind

In [14]:


glucose_pattern = r"(?<=Fasting Glucose = )(?P<glucose>\d+(?:\.\d+)?)"

match = re.search(glucose_pattern, clinical_report)

if match:
    print(match.groupdict())
else:
    print("Glucose not found")

{'glucose': '126'}


## Step 9 Diagnosis Extraction with Alternation

In [15]:


diagnosis_pattern = (
    r"Diagnosis\s*[:=]\s*"
    r"(?P<diagnosis>"
    r"Type 2 Diabetes Mellitus"
    r"|Type 1 Diabetes Mellitus"
    r"|Diabetes Mellitus"
    r"|Hypertension"
    r"|Asthma"
    r")"
)

match = re.search(diagnosis_pattern, clinical_report, re.IGNORECASE)

if match:
    print(match.groupdict())
else:
    print("Diagnosis not found")

{'diagnosis': 'Type 2 Diabetes Mellitus'}


## Step 10  Extract Medication Information

In [16]:


medication_pattern = (
    r"(?P<medication>Metformin|Atorvastatin)"
    r"\s+"
    r"(?P<dose>\d+\s*mg)"
    r"\s+"
    r"(?P<frequency>once|twice)"
    r"\s+daily"
)

matches = re.finditer(
    medication_pattern,
    clinical_report,
    re.IGNORECASE
)

for match in matches:
    print(match.groupdict())

{'medication': 'Metformin', 'dose': '500 mg', 'frequency': 'twice'}
{'medication': 'Atorvastatin', 'dose': '20 mg', 'frequency': 'once'}


## Step 11  Build Structured Clinical Record

In [17]:


record = {
    "patient_name": "John Smith",
    "mrn": "PT-20481",
    "age": 47,
    "sex": "Male",
    "blood_pressure": "138/88",
    "heart_rate": 82,
    "temperature": 98.6,
    "respiratory_rate": 18,
    "fasting_glucose": 126,
    "hba1c": 6.7,
    "total_cholesterol": 212,
    "diagnosis": "Type 2 Diabetes Mellitus",
    "medications": [
        {
            "medication": "Metformin",
            "dose": "500 mg",
            "frequency": "twice daily"
        },
        {
            "medication": "Atorvastatin",
            "dose": "20 mg",
            "frequency": "once daily"
        }
    ]
}

print(record)

{'patient_name': 'John Smith', 'mrn': 'PT-20481', 'age': 47, 'sex': 'Male', 'blood_pressure': '138/88', 'heart_rate': 82, 'temperature': 98.6, 'respiratory_rate': 18, 'fasting_glucose': 126, 'hba1c': 6.7, 'total_cholesterol': 212, 'diagnosis': 'Type 2 Diabetes Mellitus', 'medications': [{'medication': 'Metformin', 'dose': '500 mg', 'frequency': 'twice daily'}, {'medication': 'Atorvastatin', 'dose': '20 mg', 'frequency': 'once daily'}]}


## Step 12  Automatic Clinical Record Parser

In [18]:


def parse_clinical_report(text):

    record = {}

    # Patient information
    patient_match = re.search(
        r"Patient:\s*(?P<patient_name>[A-Za-z ]+)",
        text
    )

    mrn_match = re.search(
        r"MRN:\s*(?P<mrn>[A-Z]{2}-\d+)",
        text
    )

    demographic_match = re.search(
        r"Age:\s*(?P<age>\d+)\s*years\s*\|\s*"
        r"Sex:\s*(?P<sex>[A-Za-z]+)",
        text
    )

    # Vital signs
    bp_match = re.search(
        r"BP\s*[:=]?\s*(?P<bp>\d{2,3}/\d{2,3})\s*mmHg",
        text
    )

    hr_match = re.search(
        r"Heart Rate\s*[:=]?\s*(?P<heart_rate>\d{2,3})\s*bpm",
        text
    )

    temp_match = re.search(
        r"Temperature\s*[:=]?\s*(?P<temperature>\d+(?:\.\d+)?)\s*F",
        text
    )

    rr_match = re.search(
        r"Respiratory Rate\s*[:=]?\s*(?P<respiratory_rate>\d{1,2})\s*/min",
        text
    )

    # Diagnosis
    diagnosis_match = re.search(
        r"Diagnosis\s*[:=]\s*(?P<diagnosis>"
        r"Type 2 Diabetes Mellitus"
        r"|Type 1 Diabetes Mellitus"
        r"|Diabetes Mellitus"
        r"|Hypertension"
        r"|Asthma"
        r")",
        text,
        re.IGNORECASE
    )

    # Add extracted values
    if patient_match:
        record["patient_name"] = patient_match.group("patient_name").strip()

    if mrn_match:
        record["mrn"] = mrn_match.group("mrn")

    if demographic_match:
        record["age"] = int(demographic_match.group("age"))
        record["sex"] = demographic_match.group("sex")

    if bp_match:
        record["blood_pressure"] = bp_match.group("bp")

    if hr_match:
        record["heart_rate"] = int(hr_match.group("heart_rate"))

    if temp_match:
        record["temperature"] = float(temp_match.group("temperature"))

    if rr_match:
        record["respiratory_rate"] = int(
            rr_match.group("respiratory_rate")
        )

    if diagnosis_match:
        record["diagnosis"] = diagnosis_match.group("diagnosis")

    return record


parsed_record = parse_clinical_report(clinical_report)

print(parsed_record)

{'patient_name': 'John Smith', 'mrn': 'PT-20481', 'age': 47, 'sex': 'Male', 'blood_pressure': '138/88', 'heart_rate': 82, 'temperature': 98.6, 'respiratory_rate': 18, 'diagnosis': 'Type 2 Diabetes Mellitus'}


## Step 13 Add Laboratory Results Automatically

In [19]:


lab_pattern = (
    r"(?P<test>Fasting Glucose|HbA1c|Total Cholesterol)"
    r"\s*[:=]\s*"
    r"(?P<value>\d+(?:\.\d+)?)"
    r"\s*(?P<unit>mg/dL|%)"
)

lab_results = {}

for match in re.finditer(lab_pattern, clinical_report, re.IGNORECASE):
    test = match.group("test")
    value = match.group("value")
    unit = match.group("unit")

    lab_results[test] = {
        "value": float(value),
        "unit": unit
    }

print(lab_results)

{'Fasting Glucose': {'value': 126.0, 'unit': 'mg/dL'}, 'HbA1c': {'value': 6.7, 'unit': '%'}, 'Total Cholesterol': {'value': 212.0, 'unit': 'mg/dL'}}


## Step 14  Extract Medications Automatically

In [20]:


medication_pattern = (
    r"(?P<medication>Metformin|Atorvastatin)"
    r"\s+"
    r"(?P<dose>\d+\s*mg)"
    r"\s+"
    r"(?P<frequency>once|twice)"
    r"\s+daily"
)

medications = []

for match in re.finditer(
    medication_pattern,
    clinical_report,
    re.IGNORECASE
):
    medications.append({
        "medication": match.group("medication"),
        "dose": match.group("dose"),
        "frequency": match.group("frequency") + " daily"
    })

print(medications)

[{'medication': 'Metformin', 'dose': '500 mg', 'frequency': 'twice daily'}, {'medication': 'Atorvastatin', 'dose': '20 mg', 'frequency': 'once daily'}]


## Step 16  Create a Messier Clinical Report

In [21]:


messy_report = """
PATIENT INFORMATION
Patient : Sarah Khan
MRN=PK-8831
Age: 52 years | Sex: Female

VITAL SIGNS
BP: 145/92 mmHg
Heart Rate = 91 bpm
Temperature: 99.1 F
Respiratory Rate 20/min

LABORATORY RESULTS
Fasting Glucose: 141 mg/dL
HbA1c = 7.2 %
Total Cholesterol: 228 mg/dL

CLINICAL ASSESSMENT
Diagnosis = Type 2 Diabetes Mellitus

MEDICATIONS
Metformin 850 mg once daily
"""

print(messy_report)


PATIENT INFORMATION
Patient : Sarah Khan
MRN=PK-8831
Age: 52 years | Sex: Female

VITAL SIGNS
BP: 145/92 mmHg
Heart Rate = 91 bpm
Temperature: 99.1 F
Respiratory Rate 20/min

LABORATORY RESULTS
Fasting Glucose: 141 mg/dL
HbA1c = 7.2 %
Total Cholesterol: 228 mg/dL

CLINICAL ASSESSMENT
Diagnosis = Type 2 Diabetes Mellitus

MEDICATIONS
Metformin 850 mg once daily



## Step 17  Test Current Parser on Messy Report

In [22]:


messy_record = parse_clinical_report(messy_report)

print(messy_record)

{'age': 52, 'sex': 'Female', 'blood_pressure': '145/92', 'heart_rate': 91, 'temperature': 99.1, 'respiratory_rate': 20, 'diagnosis': 'Type 2 Diabetes Mellitus'}


## Step 18  More Flexible Patient and MRN Regex

In [23]:


patient_pattern = r"Patient\s*[:=]\s*(?P<patient_name>[A-Za-z ]+)"
mrn_pattern = r"MRN\s*[:=]\s*(?P<mrn>[A-Z]{2}-\d+)"

patient_match = re.search(patient_pattern, messy_report)
mrn_match = re.search(mrn_pattern, messy_report)

if patient_match:
    print("Patient:", patient_match.group("patient_name").strip())

if mrn_match:
    print("MRN:", mrn_match.group("mrn"))

Patient: Sarah Khan
MRN: PK-8831


## Step 19 Multiple Clinical Reports

In [24]:


clinical_reports = [
    clinical_report,
    messy_report
]

parsed_records = []

for report in clinical_reports:
    parsed_records.append(
        parse_clinical_report(report)
    )

for i, record in enumerate(parsed_records, start=1):
    print(f"Report {i}:")
    print(record)
    print()

Report 1:
{'patient_name': 'John Smith', 'mrn': 'PT-20481', 'age': 47, 'sex': 'Male', 'blood_pressure': '138/88', 'heart_rate': 82, 'temperature': 98.6, 'respiratory_rate': 18, 'diagnosis': 'Type 2 Diabetes Mellitus'}

Report 2:
{'age': 52, 'sex': 'Female', 'blood_pressure': '145/92', 'heart_rate': 91, 'temperature': 99.1, 'respiratory_rate': 20, 'diagnosis': 'Type 2 Diabetes Mellitus'}



## Step 20  Final Robust Parser

In [25]:

def parse_clinical_report(text):

    record = {}

    # Patient information
    patient_match = re.search(
        r"Patient\s*[:=]\s*(?P<patient_name>[A-Za-z ]+)",
        text,
        re.IGNORECASE
    )

    mrn_match = re.search(
        r"MRN\s*[:=]\s*(?P<mrn>[A-Z]{2}-\d+)",
        text,
        re.IGNORECASE
    )

    demographic_match = re.search(
        r"Age\s*:\s*(?P<age>\d+)\s*years\s*\|\s*"
        r"Sex\s*:\s*(?P<sex>[A-Za-z]+)",
        text,
        re.IGNORECASE
    )

    # Vital signs
    bp_match = re.search(
        r"BP\s*[:=]?\s*(?P<bp>\d{2,3}/\d{2,3})\s*mmHg",
        text,
        re.IGNORECASE
    )

    hr_match = re.search(
        r"Heart Rate\s*[:=]?\s*(?P<heart_rate>\d{2,3})\s*bpm",
        text,
        re.IGNORECASE
    )

    temp_match = re.search(
        r"Temperature\s*[:=]?\s*(?P<temperature>\d+(?:\.\d+)?)\s*F",
        text,
        re.IGNORECASE
    )

    rr_match = re.search(
        r"Respiratory Rate\s*[:=]?\s*(?P<respiratory_rate>\d{1,2})\s*/min",
        text,
        re.IGNORECASE
    )

    # Diagnosis
    diagnosis_match = re.search(
        r"Diagnosis\s*[:=]\s*(?P<diagnosis>"
        r"Type 2 Diabetes Mellitus"
        r"|Type 1 Diabetes Mellitus"
        r"|Diabetes Mellitus"
        r"|Hypertension"
        r"|Asthma"
        r")",
        text,
        re.IGNORECASE
    )

    # Laboratory results
    lab_pattern = (
        r"(?P<test>Fasting Glucose|HbA1c|Total Cholesterol)"
        r"\s*[:=]\s*"
        r"(?P<value>\d+(?:\.\d+)?)"
        r"\s*(?P<unit>mg/dL|%)"
    )

    lab_results = {}

    for match in re.finditer(
        lab_pattern,
        text,
        re.IGNORECASE
    ):
        lab_results[match.group("test")] = {
            "value": float(match.group("value")),
            "unit": match.group("unit")
        }

    # Medications
    medication_pattern = (
        r"(?P<medication>Metformin|Atorvastatin)"
        r"\s+"
        r"(?P<dose>\d+\s*mg)"
        r"\s+"
        r"(?P<frequency>once|twice)"
        r"\s+daily"
    )

    medications = []

    for match in re.finditer(
        medication_pattern,
        text,
        re.IGNORECASE
    ):
        medications.append({
            "medication": match.group("medication"),
            "dose": match.group("dose"),
            "frequency": match.group("frequency") + " daily"
        })

    # Store results
    if patient_match:
        record["patient_name"] = patient_match.group("patient_name").strip()

    if mrn_match:
        record["mrn"] = mrn_match.group("mrn")

    if demographic_match:
        record["age"] = int(demographic_match.group("age"))
        record["sex"] = demographic_match.group("sex")

    if bp_match:
        record["blood_pressure"] = bp_match.group("bp")

    if hr_match:
        record["heart_rate"] = int(hr_match.group("heart_rate"))

    if temp_match:
        record["temperature"] = float(temp_match.group("temperature"))

    if rr_match:
        record["respiratory_rate"] = int(
            rr_match.group("respiratory_rate")
        )

    if diagnosis_match:
        record["diagnosis"] = diagnosis_match.group("diagnosis")

    record["lab_results"] = lab_results
    record["medications"] = medications

    return record


# Final test on both reports

for i, report in enumerate(
    [clinical_report, messy_report],
    start=1
):
    print(f"Report {i}:")
    print(parse_clinical_report(report))
    print()

Report 1:
{'patient_name': 'John Smith', 'mrn': 'PT-20481', 'age': 47, 'sex': 'Male', 'blood_pressure': '138/88', 'heart_rate': 82, 'temperature': 98.6, 'respiratory_rate': 18, 'diagnosis': 'Type 2 Diabetes Mellitus', 'lab_results': {'Fasting Glucose': {'value': 126.0, 'unit': 'mg/dL'}, 'HbA1c': {'value': 6.7, 'unit': '%'}, 'Total Cholesterol': {'value': 212.0, 'unit': 'mg/dL'}}, 'medications': [{'medication': 'Metformin', 'dose': '500 mg', 'frequency': 'twice daily'}, {'medication': 'Atorvastatin', 'dose': '20 mg', 'frequency': 'once daily'}]}

Report 2:
{'patient_name': 'Sarah Khan', 'mrn': 'PK-8831', 'age': 52, 'sex': 'Female', 'blood_pressure': '145/92', 'heart_rate': 91, 'temperature': 99.1, 'respiratory_rate': 20, 'diagnosis': 'Type 2 Diabetes Mellitus', 'lab_results': {'Fasting Glucose': {'value': 141.0, 'unit': 'mg/dL'}, 'HbA1c': {'value': 7.2, 'unit': '%'}, 'Total Cholesterol': {'value': 228.0, 'unit': 'mg/dL'}}, 'medications': [{'medication': 'Metformin', 'dose': '850 mg', 'f

## Step 21  Final Structured DataFrame

In [26]:


parsed_records = [
    parse_clinical_report(clinical_report),
    parse_clinical_report(messy_report)
]

df = pd.DataFrame(parsed_records)

print("Shape:", df.shape)
display(df)

Shape: (2, 11)


,patient_name,mrn,age,sex,blood_pressure,heart_rate,temperature,respiratory_rate,diagnosis,lab_results,medications
0,John Smith,PT-20481,47,Male,138/88,82,98.6,18,Type 2 Diabetes Mellitus,"{'Fasting Glucose': {'value': 126.0, 'unit': '...","[{'medication': 'Metformin', 'dose': '500 mg',..."
1,Sarah Khan,PK-8831,52,Female,145/92,91,99.1,20,Type 2 Diabetes Mellitus,"{'Fasting Glucose': {'value': 141.0, 'unit': '...","[{'medication': 'Metformin', 'dose': '850 mg',..."


## Step 22  Final Validation

In [28]:


required_fields = [
    "patient_name",
    "mrn",
    "age",
    "sex",
    "blood_pressure",
    "heart_rate",
    "temperature",
    "respiratory_rate",
    "diagnosis",
    "lab_results",
    "medications"
]

print("Missing values:")
print(df[required_fields].isna().sum())

print("\nNumber of records:", len(df))

print(
    "Complete records:",
    df[required_fields].notna().all(axis=1).sum()
)

Missing values:
patient_name        0
mrn                 0
age                 0
sex                 0
blood_pressure      0
heart_rate          0
temperature         0
respiratory_rate    0
diagnosis           0
lab_results         0
medications         0
dtype: int64

Number of records: 2
Complete records: 2


# Conclusion

In this project, Regex was used to parse realistic and messy clinical reports into structured data. Named groups, alternation, lookahead, lookbehind, flexible patterns, and **re.finditer()** were used to extract patient information, vital signs, laboratory results, diagnoses, and medications.

The parser was tested on multiple clinical reports and successfully handled different formats such as **:** and **=** separators. The extracted information was converted into a structured DataFrame and final validation confirmed that all 2 records were complete with no missing required fields.

This project demonstrates how Python Regex can be used for basic clinical text processing and structured information extraction.
